# Quick-train and predict

End-to-end example that loads a CSV of polymer data, k-fold splits it, trains a small message-passing GNN, and runs predictions on held-out data. Designed to run in a few minutes on CPU or a single GPU.

Inputs and outputs:
- input file: `../data/stereopolymer_input_nopush.csv` (bundled with the repo)
- saved models: `./quick_train_models/`
- saved kfold split: `./quick_train_models/kfold_splits.json`

Required environment: TensorFlow 2.20 / Keras 3 (e.g. `conda activate /home/jlaw/.conda-envs/prekd_py312_tf220`).

## 1. Imports and parameters

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

import tensorflow as tf
import keras
import nfp

from polyid import MultiModel, Parameters
from polyid.preprocessors import PolymerPreprocessor
from polyid.models import global100
from nfp.preprocessing.features import atom_features_v1, bond_features_v1

print(f"tensorflow {tf.__version__}, keras {keras.__version__}, nfp {nfp.__version__}")

2026-06-24 11:55:58.502484: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-24 11:56:00.317048: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


tensorflow 2.20.0, keras 3.13.2, nfp 0.3.12+4.g8500673


In [2]:
# Small/fast hyperparameters for a quick example.
# For production training, increase epochs (>= 500), num_messages (>= 8), and feature sizes (>= 128).
params = Parameters()
params.batch_size = 16
params.learning_rate = 1e-3
params.decay = 1e-5
params.atom_features = 32
params.bond_features = 32
params.num_messages = 4
params.dropout = 0.05
params.epochs = 25
params.kfolds = 3
params.prediction_columns = ["Tg", "Tm"]

save_folder = Path("./quick_train_models")
save_folder.mkdir(parents=True, exist_ok=True)

data_csv = "../data/stereopolymer_input_nopush.csv"

## 2. Load and inspect the dataset

The bundled CSV has one row per replicate polymer structure. Required columns: `smiles_polymer`, `smiles_monomer`, `monomers`, `distribution`, `replicate_structure`, `mechanism`, and one column per property in `params.prediction_columns`.

In [3]:
df = pd.read_csv(data_csv)
print(f"shape: {df.shape}")
df.head(3)

shape: (229, 12)


,smiles_monomer,pm,polymer_name,Tg,Tm,Tm_units,Tg_units,monomers,distribution,replicate_structure,smiles_polymer,mechanism
0,C=C(C(=O)O)[C@@H](C)O.C=C(C(=O)O)[C@H](C)O,0.71,poly(3-hydroxy-2-methylenebutanoic acid),NaN,80.5,C,C,"('C=C(C(=O)O)[C@@H](C)O', 'C=C(C(=O)O)[C@H](C)O')",[],0,C=C(C(=O)O[C@@H](C)C(=C)C(=O)O[C@@H](C)C(=C)C(...,ester
1,C[C@@H](O)CC(=O)O.C[C@H](O)CC(=O)O,0.96,poly(RS-3-hydroxybutyrate),6.0,154.5,C,C,"('C[C@@H](O)CC(=O)O', 'C[C@H](O)CC(=O)O')",[],0,C[C@@H](O)CC(=O)O[C@H](C)CC(=O)O[C@H](C)CC(=O)...,ester
2,C[C@@H](O)CC(=O)O.C[C@H](O)CC(=O)O,0.95,poly(RS-3-hydroxybutyrate),4.0,147.0,C,C,"('C[C@@H](O)CC(=O)O', 'C[C@H](O)CC(=O)O')",[],0,C[C@@H](O)CC(=O)O[C@H](C)CC(=O)O[C@H](C)CC(=O)...,ester


In [4]:
df[params.prediction_columns].describe()

,Tg,Tm
count,155.000000,123.000000
mean,24.546667,143.414131
std,26.470797,40.499473
min,-30.800000,49.000000
25%,4.000000,110.000000
50%,36.000000,150.000000
75%,44.000000,173.000000
max,91.000000,212.000000


## 6. Reload models and predict on the holdout set

`MultiModel.load_models` reads every `model_*` subfolder. We call `make_predictions` (one row per k-fold model x holdout polymer) and then average the per-fold predictions per polymer.

In [5]:
rng = np.random.default_rng(seed=42)
unique_monomers = df.smiles_monomer.unique()
holdout_idx = rng.choice(len(unique_monomers), size=max(1, int(0.15 * len(unique_monomers))), replace=False)
holdout_monomers = set(unique_monomers[holdout_idx])

df_train = df[~df.smiles_monomer.isin(holdout_monomers)].reset_index(drop=True)
df_holdout = df[df.smiles_monomer.isin(holdout_monomers)].reset_index(drop=True)

train_csv = save_folder / "df_train.csv"
holdout_csv = save_folder / "df_holdout.csv"
df_train.to_csv(train_csv)
df_holdout.to_csv(holdout_csv)
print(f"train rows: {len(df_train)} | holdout rows: {len(df_holdout)}")

train rows: 197 | holdout rows: 32


## 4. Build the MultiModel and k-fold split

In [6]:
mm = MultiModel()
mm.load_dataset(str(train_csv), prediction_columns=params.prediction_columns)

# Stratify by `mechanism` (use stratify=False if your data has only one mechanism class)
stratify = df_train.mechanism.nunique() > 1
mm.split_data(
    kfolds=params.kfolds,
    stratify=stratify,
    kfold_save=str(save_folder / "kfold_splits.json"),
)

# Fit a RobustScaler on the prediction columns for each SingleModel
mm.generate_data_scaler()

# Generate the per-model SMILES preprocessor
mm.generate_preprocessors(
    preprocessor=PolymerPreprocessor,
    atom_features=atom_features_v1,
    bond_features=bond_features_v1,
    batch_size=params.batch_size,
)

print(f"split into {len(mm.models)} kfolds")
print({i: (len(m.df_train), len(m.df_validate)) for i, m in enumerate(mm.models)})

  0%|          | 0/125 [00:00<?, ?it/s]

2026-06-24 11:56:25.982131: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2026-06-24 11:56:25.989070: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:160] env: CUDA_VISIBLE_DEVICES=""
2026-06-24 11:56:25.993569: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:163] CUDA_VISIBLE_DEVICES is set to an empty string - this hides all GPUs from CUDA
2026-06-24 11:56:25.993592: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:171] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
2026-06-24 11:56:25.994897: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:176] retrieving CUDA diagnostic information for host: kd5
2026-06-24 11:56:25.994920: I external/local_xla/xla/stream_executor/cud

  1%|          | 1/125 [00:00<01:57,  1.05it/s]al_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:194] kernel reported version is: 545.23.8
2026-06-24 11:56:25.999080: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:284] kernel version seems to match DSO: 545.23.8


 14%|█▎        | 17/125 [00:01<00:05, 18.95it/s]

 31%|███       | 39/125 [00:01<00:01, 46.84it/s]

...

 42%|████▏     | 57/136 [00:00<00:00, 281.05it/s]

 63%|██████▎   | 86/136 [00:00<00:00, 273.89it/s]

 84%|████████▍ | 114/136 [00:00<00:00, 202.89it/s]

100%|██████████| 136/136 [00:00<00:00, 232.69it/s]

split into 3 kfolds
{0: (125, 72), 1: (133, 64), 2: (136, 61)}


## 5. Train all k-fold models

Each model is saved as `quick_train_models/model_{i}/model_{i}.keras` plus a `model_{i}_data.pk` with the preprocessor, scaler, and training-time dataframes.

In [7]:
mm.train_models(
    modelbuilder=global100,
    model_params=params.to_dict(),
    save_folder=str(save_folder),
    save_training=True,
    verbose=2,
)

<bound method Model.summary of <Functional name=functional, built=True>>
Epoch 1/25


2026-06-24 11:56:36.309278: E tensorflow/core/util/util.cc:131] oneDNN supports DT_INT32 only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


2026-06-24 11:56:38.354463: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
/home/jlaw/.conda-envs/prekd_py312_tf220/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()



Epoch 1: val_loss improved from None to 0.74153, saving model to quick_train_models/model_0/model_0.keras


2026-06-24 11:56:39.469857: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
/home/jlaw/.conda-envs/prekd_py312_tf220/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


...

47/61 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step

51/61 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step

55/61 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step

58/61 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step

61/61 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step


In [8]:
# Per-fold validation MAE (in scaled units) over the last 5 epochs
loss_log = mm.df_loss_log
loss_log.groupby("model_id").tail(5).groupby("model_id")[["loss", "val_loss"]].mean().round(4)

,loss,val_loss
model_id,,
0,0.2675,0.3109
1,0.2396,0.2568
2,0.3555,0.4770


## 6. Reload models and predict on the holdout set

`MultiModel.load_models` reads every `model_*` subfolder. We call `make_predictions` (one row per k-fold model x holdout polymer) and then average the per-fold predictions per polymer.

In [9]:
mm_loaded = MultiModel.load_models(str(save_folder))

# Drop string `_units` columns so the per-polymer average is purely numeric
df_holdout_for_pred = df_holdout.drop(
    columns=[c for c in df_holdout.columns if c.endswith('_units')]
)

df_pred = mm_loaded.make_predictions(df_holdout_for_pred)
print(f'raw predictions: {len(df_pred)} rows ({params.kfolds} kfolds x {len(df_holdout_for_pred)} polymers)')
df_pred.head()

Making predictions for 32 with batch_size=1


 1/32 ━━━━━━━━━━━━━━━━━━━━ 14s 455ms/step

 5/32 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step  

 9/32 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step

13/32 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step

...

25/32 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step

30/32 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step

32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step


raw predictions: 96 rows (3 kfolds x 32 polymers)


,smiles_monomer,pm,polymer_name,Tg,Tm,monomers,distribution,replicate_structure,smiles_polymer,mechanism,Tg_pred,Tm_pred,model_id
0,C[C@@H](O)CC(=O)O.C[C@H](O)CC(=O)O,0.96,poly(RS-3-hydroxybutyrate),6.000000,154.5,"('C[C@@H](O)CC(=O)O', 'C[C@H](O)CC(=O)O')",[],0,C[C@@H](O)CC(=O)O[C@H](C)CC(=O)O[C@H](C)CC(=O)...,ester,5.899467,138.842102,2
1,C[C@@H](O)CC(=O)O.C[C@H](O)CC(=O)O,0.95,poly(RS-3-hydroxybutyrate),4.000000,147.0,"('C[C@@H](O)CC(=O)O', 'C[C@H](O)CC(=O)O')",[],0,C[C@@H](O)CC(=O)O[C@H](C)CC(=O)O[C@H](C)CC(=O)...,ester,5.899467,138.842102,2
2,C[C@@H](O)CC(=O)O.C[C@H](O)CC(=O)O,0.94,poly(RS-3-hydroxybutyrate),4.000000,136.0,"('C[C@@H](O)CC(=O)O', 'C[C@H](O)CC(=O)O')",[],0,C[C@@H](O)CC(=O)O[C@H](C)CC(=O)O[C@H](C)CC(=O)...,ester,5.899467,138.842102,2
3,C[C@@H](O)CC(=O)O.C[C@H](O)CC(=O)O,0.93,poly(RS-3-hydroxybutyrate),2.666667,139.5,"('C[C@@H](O)CC(=O)O', 'C[C@H](O)CC(=O)O')",[],0,C[C@H](O)CC(=O)O[C@@H](C)CC(=O)O[C@@H](C)CC(=O...,ester,5.899467,138.842102,2
4,C[C@@H](O)CC(=O)O.C[C@H](O)CC(=O)O,0.91,poly(RS-3-hydroxybutyrate),-1.000000,128.0,"('C[C@@H](O)CC(=O)O', 'C[C@H](O)CC(=O)O')",[],0,C[C@@H](O)CC(=O)O[C@H](C)CC(=O)O[C@H](C)CC(=O)...,ester,5.899467,138.842102,2


In [10]:
# Average the per-fold predictions per polymer
pred_cols = [f'{c}_pred' for c in params.prediction_columns]
df_avg = (
    df_pred.groupby('smiles_polymer')[pred_cols].mean()
    .merge(df_holdout[['smiles_polymer'] + params.prediction_columns].drop_duplicates('smiles_polymer'),
           on='smiles_polymer')
)

# Quick MAE on the holdout
for col in params.prediction_columns:
    mask = df_avg[col].notna() & df_avg[f'{col}_pred'].notna()
    if mask.sum() > 0:
        mae = (df_avg.loc[mask, col] - df_avg.loc[mask, f'{col}_pred']).abs().mean()
        print(f'{col}: n={mask.sum():3d}, holdout MAE = {mae:.2f}')
df_avg.head()

Tg: n=  8, holdout MAE = 8.08
Tm: n= 12, holdout MAE = 65.82


,smiles_polymer,Tg_pred,Tm_pred,Tg,Tm
0,C=CCOC(=O)[C@@H](O)CC(=O)O[C@H](CC(=O)O[C@@H](...,36.383049,164.606812,NaN,49.0
1,C=CCOC(=O)[C@@H](O)CC(=O)O[C@H](CC(=O)O[C@@H](...,36.383049,164.606812,NaN,80.0
2,C=CCOC(=O)[C@@H](O)CC(=O)O[C@H](CC(=O)O[C@@H](...,36.383049,164.606812,NaN,80.0
3,C=CCOC(=O)[C@@H](O)CC(=O)O[C@H](CC(=O)O[C@H](C...,36.383049,164.606812,NaN,51.0
4,C=CCOC(=O)[C@H](CC(=O)O)OC(=O)C[C@@H](OC(=O)C[...,36.383053,164.606812,NaN,51.0
